# 13 — Installing and Using the WDO Library

In Module 12 you saw how to pull repeated notebook code into a `.py` module.  
That module already exists in this project at `src/wdo/`.  
This lesson shows you how to **install it** and **import from it** so you never have to paste those functions again.

---
## 1 — The `src` Layout

The project is organized like this:

```
02-Missile_Geometry_202/
  pyproject.toml          ← packaging metadata
  src/
    wdo/
      __init__.py         ← makes wdo importable as a package
      geo.py              ← geodetic math (distance, bearing, destination …)
      maps.py             ← ipyleaflet helper wrappers
      io_shapefile.py     ← shapefile reader (no GeoPandas)
      simulate_threats.py ← threat data generator
```

The `src/` folder is a deliberate design choice.  
Putting the package one level down means Python will only find `wdo` if it has been **installed** — not just because you happen to be running a notebook from the same directory.  
That forces you to do the install step, which is good practice.

---
## 2 — Reading `pyproject.toml`

Open `pyproject.toml` in the project root.  It looks like this:

```toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "wdo"
version = "0.0.0"
description = "WDO helper package for Missile Geometry lessons"
requires-python = ">=3.10"

[tool.setuptools.packages.find]
where = ["src"]
```

| Section | What it does |
|---------|-------------|
| `[build-system]` | Tells pip *how* to build the package (uses `setuptools`) |
| `[project]` | The package's identity: name, version, Python requirement |
| `packages.find` | Tells setuptools to look for packages inside `src/` |

Most real-world Python packages have a file that looks exactly like this.

---
## 3 — Installing with `pip install -e .`

Run the cell below **once**.  
The `-e` flag means **editable** — if you later change a file inside `src/wdo/`, the change is immediately live without reinstalling.

In [ ]:
# Run from the project root (02-Missile_Geometry_202/)
# The ! prefix sends the command to your shell instead of Python
import subprocess, pathlib

project_root = pathlib.Path.cwd()
# Walk up until we find pyproject.toml
for p in [project_root, *project_root.parents]:
    if (p / "pyproject.toml").exists():
        project_root = p
        break

print(f"Installing from: {project_root}")
result = subprocess.run(
    ["pip", "install", "-e", "."],
    cwd=project_root,
    capture_output=True,
    text=True,
)
print(result.stdout[-800:] if len(result.stdout) > 800 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-400:])

In [ ]:
# Confirm the install worked
import wdo
print("wdo imported successfully")
print("wdo location:", wdo.__file__)

If you see a path ending in `src/wdo/__init__.py`, the editable install is working correctly.

> **What is `__init__.py`?**  
> It is the file Python runs when you write `import wdo`.  
> It controls what names are available at the top level of the package.  
> Open `src/wdo/__init__.py` to see what is exported.

---
## 4 — What's in `wdo`

The library has three main modules.  Import from them directly.

In [ ]:
# --- wdo.geo: geodetic math ---
from wdo.geo import (
    LatLon,
    haversine_km,
    initial_bearing_deg,
    destination_point,
    bbox_latlon,
    segments_intersect,
    point_in_polygon,
    point_feature,
    line_feature,
    polygon_feature,
    feature_collection,
)

# --- wdo.maps: ipyleaflet helpers ---
from wdo.maps import map as wdo_map, add_marker, add_geojson, fit_bbox

print("All imports succeeded.")

---
## 5 — `wdo.geo` in Practice

Every geometry calculation you have written across Modules 05–11 is now one import away.  
Let's walk through the main functions.

### 5a — Distance and Bearing

In [ ]:
# Two sites — Sheppard AFB and Tinker AFB
sheppard = LatLon(lat=33.9889, lon=-98.4912)
tinker   = LatLon(lat=35.4147, lon=-97.3867)

dist = haversine_km(sheppard.lat, sheppard.lon, tinker.lat, tinker.lon)
bearing = initial_bearing_deg(sheppard.lat, sheppard.lon, tinker.lat, tinker.lon)

print(f"Distance:  {dist:.1f} km")
print(f"Bearing:   {bearing:.1f}°")

### 5b — Destination Point

`destination_point` is the inverse of Haversine: given a start, a bearing, and a distance, where do you end up?

In [ ]:
# A threat launched from Sheppard, heading 045° (northeast), after 200 km
endpoint = destination_point(sheppard.lat, sheppard.lon, bearing_deg=45, distance_km=200)
print(f"Endpoint: lat={endpoint.lat:.4f}, lon={endpoint.lon:.4f}")

### 5c — Bounding Box

In [ ]:
sites = [
    LatLon(33.9889, -98.4912),   # Sheppard
    LatLon(35.4147, -97.3867),   # Tinker
    LatLon(32.3838, -97.4197),   # NAS Fort Worth JRB
    LatLon(29.5338, -98.5811),   # Lackland
]

min_lat, min_lon, max_lat, max_lon = bbox_latlon(sites)
print(f"Bounding box:")
print(f"  South: {min_lat:.4f}  North: {max_lat:.4f}")
print(f"  West:  {min_lon:.4f}  East:  {max_lon:.4f}")

### 5d — Segment Intersection and Point-in-Polygon

In [ ]:
# Does a threat flight path cross a defense perimeter segment?
threat_start = (32.0, -99.0)
threat_end   = (34.5, -96.5)

perimeter_a  = (33.0, -99.5)
perimeter_b  = (33.0, -96.0)

crossed = segments_intersect(threat_start, threat_end, perimeter_a, perimeter_b)
print(f"Threat crosses perimeter: {crossed}")


# Is a point inside a defense zone polygon?
zone = [
    (32.0, -99.0),
    (32.0, -96.0),
    (36.0, -96.0),
    (36.0, -99.0),
]

impact_point = (34.0, -97.5)
inside = point_in_polygon(impact_point, zone)
print(f"Impact point inside zone: {inside}")

---
## 6 — Building GeoJSON with `wdo.geo`

The GeoJSON builder helpers let you construct displayable features without writing raw dicts.

In [ ]:
import json

# Build a small feature collection of the four sites
features = [
    point_feature(s.lon, s.lat, {"name": name})
    for s, name in zip(sites, ["Sheppard", "Tinker", "NAS Fort Worth", "Lackland"])
]

# Add a line connecting Sheppard → Tinker
features.append(
    line_feature(
        [[sheppard.lon, sheppard.lat], [tinker.lon, tinker.lat]],
        props={"label": "Sheppard–Tinker", "distance_km": round(dist, 1)},
    )
)

fc = feature_collection(features)
print(json.dumps(fc, indent=2)[:600], "\n...")

---
## 7 — Displaying with `wdo.maps`

`wdo.maps` wraps `ipyleaflet` to reduce the setup boilerplate you wrote in Modules 02–04.

In [ ]:
# One call creates a map centered on the bounding box
m = wdo_map(
    center=((min_lat + max_lat) / 2, (min_lon + max_lon) / 2),
    zoom=7,
    basemap_name="positron",
)
m

In [ ]:
# Add all the features in one call
add_geojson(
    m,
    data=fc,
    style={"color": "#c0392b", "weight": 2},
    hover_style={"color": "#e74c3c", "weight": 4},
)

# Fit the map to the bounding box
# bbox_latlon returns (min_lat, min_lon, max_lat, max_lon)
# fit_bbox expects [min_lon, min_lat, max_lon, max_lat]
fit_bbox(m, [min_lon, min_lat, max_lon, max_lat])

---
## 8 — Before and After

Here is the same map setup written both ways:

In [ ]:
# BEFORE (Module 02 style) — ~15 lines of boilerplate
from ipyleaflet import Map, GeoJSON, ScaleControl, basemaps
from ipywidgets import Layout

m_old = Map(
    center=(33.5, -97.8),
    zoom=7,
    basemap=basemaps.CartoDB.Positron,
    scroll_wheel_zoom=True,
    layout=Layout(width="100%", height="600px"),
)
m_old.add(ScaleControl(position="bottomleft"))
layer = GeoJSON(data=fc, style={"color": "#c0392b", "weight": 2})
m_old.add(layer)
m_old

In [ ]:
# AFTER (wdo.maps style) — same result, ~5 lines
m_new = wdo_map(center=(33.5, -97.8), zoom=7)
add_geojson(m_new, data=fc, style={"color": "#c0392b", "weight": 2})
m_new

Both maps are identical.  The second version has no distracting boilerplate — the geometry and the intent are easier to read.

---
## 9 — Putting It All Together: Threat Scenario

Use `wdo` functions end-to-end: simulate a threat, compute intercept geometry, display it.

In [ ]:
from wdo.geo import destination_point, initial_bearing_deg, haversine_km, LatLon
from wdo.geo import point_feature, line_feature, feature_collection
from wdo.maps import map as wdo_map, add_geojson, add_marker

# Threat: launched from the Gulf of Mexico, heading north at 900 km/h
threat_origin  = LatLon(lat=25.0, lon=-90.0)
threat_heading = 15     # degrees (north-northeast)
threat_speed   = 900    # km/h
flight_time_hr = 1.5    # simulate 1.5 hours of flight

# Where is the threat after 1.5 hours?
threat_pos = destination_point(
    threat_origin.lat, threat_origin.lon,
    bearing_deg=threat_heading,
    distance_km=threat_speed * flight_time_hr,
)
print(f"Threat position: lat={threat_pos.lat:.3f}, lon={threat_pos.lon:.3f}")

# Defender at Sheppard
defender = LatLon(lat=33.9889, lon=-98.4912)

dist_to_threat  = haversine_km(defender.lat, defender.lon, threat_pos.lat, threat_pos.lon)
bearing_to_threat = initial_bearing_deg(defender.lat, defender.lon, threat_pos.lat, threat_pos.lon)

print(f"Distance to threat: {dist_to_threat:.1f} km")
print(f"Bearing to threat:  {bearing_to_threat:.1f}°")

In [ ]:
# Build a GeoJSON scene
features = [
    point_feature(threat_origin.lon, threat_origin.lat, {"label": "Threat Origin", "marker-color": "#e74c3c"}),
    point_feature(threat_pos.lon,    threat_pos.lat,    {"label": "Threat (t=1.5h)", "marker-color": "#e74c3c"}),
    point_feature(defender.lon,      defender.lat,      {"label": "Sheppard AFB",    "marker-color": "#2980b9"}),
    line_feature(
        [[threat_origin.lon, threat_origin.lat], [threat_pos.lon, threat_pos.lat]],
        props={"label": "Threat path", "stroke": "#e74c3c"},
    ),
    line_feature(
        [[defender.lon, defender.lat], [threat_pos.lon, threat_pos.lat]],
        props={"label": "Intercept vector", "stroke": "#2980b9"},
    ),
]

scene = feature_collection(features)

# Display
m = wdo_map(center=(30.0, -92.0), zoom=5)
add_geojson(m, data=scene,
            style={"weight": 2, "color": "#555"},
            hover_style={"weight": 4})
m

---
## 10 — Making Changes to `wdo`

Because the install is **editable** (`-e`), you can open any file under `src/wdo/` and modify it.  
After saving, just re-run your import cell — no reinstall needed.

Try it:
1. Open `src/wdo/geo.py`.
2. Add a new function `midpoint(p1: LatLon, p2: LatLon) -> LatLon` that returns the midpoint of two LatLon objects using simple averaging.
3. Import and call it in the cell below.

In [ ]:
# After adding midpoint() to geo.py, uncomment and run:
# import importlib
# import wdo.geo
# importlib.reload(wdo.geo)
# from wdo.geo import midpoint
#
# mid = midpoint(sheppard, tinker)
# print(f"Midpoint: lat={mid.lat:.4f}, lon={mid.lon:.4f}")

---
## Check Your Understanding

The `wdo/__init__.py` currently only exports five names from `wdo.maps`:

```python
from .maps import add_geojson, add_marker, bbox_to_bounds, fit_bbox, map
```

This means `from wdo import haversine_km` would **fail** — you have to write `from wdo.geo import haversine_km`.

**Question:** What would you add to `__init__.py` to make `from wdo import haversine_km, LatLon` work?  
Write the line below, and explain why you might — or might not — want to do this for every function in `wdo.geo`.

Add a line like `from .geo import haversine_km, LatLon` to `wdo/__init__.py` so those names are re-exported at the package top level. I would only expose the most common, stable APIs this way; exporting every helper clutters the namespace and makes the package surface harder to maintain.